# Projeto para Turismo - Planejador de Viagens (multiagente)

## Instalação das bibliotecas

In [1]:
!pip install crewai crewai[tools] crewai[google-genai] python-dotenv


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


## Carregamento do modelo

In [2]:
from crewai import LLM
import os

id_model = "gemini/gemini-2.5-flash" # @param {type:"string"}

gemini = LLM(
    model = id_model,
    api_key=os.getenv("GOOGLE_API_KEY")
)

## Criação das Tools (ferramentas)

In [5]:
!pip install -q duckduckgo-search ddgs langchain_community


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [6]:
from crewai.tools import tool
from langchain_community.tools import DuckDuckGoSearchResults

/tmp/ipykernel_30694/3914554391.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchResults


In [7]:
@tool
def search_web_tools(query: str):
    """
    Searches the web and returns results.
    """
    search_tool = DuckDuckGoSearchResults(num_results=4, verbose=True)
    return search_tool.run(query)

In [8]:
from langchain_community.tools import DuckDuckGoSearchRun
search = DuckDuckGoSearchRun()
search.invoke("eventos em londres no próximo mês")

'Grandes Ofertas Em Shows E Musicais Do West End De Londres. Melhores Preços Online. Aproveite As Melhores Ofertas De Última Hora Para Shows E Musicais No West End Londres. Tipos de Espectáculos, Peças de Teatro, Concertos Teatrais, Peças de teatro and more Faça suas reservas, receba confirmação e baixe seus ingressos no celular de forma prática. Londres Espetáculos e musicais: Melhores de 2026 - Cancelamento Grátis Reserve Ingressos Para Shows E Musicais Imperdíveis No West End De Londres. Garanta Seus Bilhetes Agora E Aproveite Experiências Inesquecíveis Em Londres Líder Mundial · Pagamento 100% Seguro · Guia em Português · Reserve Agora e Sem Filas Encontre eventos a acontecer a no próximo mês em Londres, Reino Unido. Explore uma variedade de atividades e interesses para planear o seu dia fora de casa perfeito. La guía más completa de qué hacer este mes en Londres: 48+ eventos — conciertos, fiestas, gastronomía, planes gratis y al aire libre. Actualizado octubre 2026.'

### Criação de classe

In [9]:
!pip install -q langchain-tavily


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [10]:
from langchain_tavily import TavilySearch
from langchain_community.tools import DuckDuckGoSearchResults

class SearchTools:
    @tool("Pesquisa na internet")
    def search_tavily(query: str = "") -> str:
        """
        Search the web using TavilyAPI.
        Recommended for more structured and recent information.
        """
        search_tavily = TavilySearch(max_results = 4)
        search_res = search_tavily.invoke(input=query)
        return search_res

    @tool("Pesquisa na internet com DuckDuckGo")
    def search_duckduckgo(query: str):
        """
        Search the web using DuckDuckGo.
        Returns a list of search results.
        """
        search_tool = DuckDuckGoSearchResults(num_results=4, verbose=True)
        return search_tool.run(query)        

In [11]:
SearchTools.search_tavily

Tool(name='Pesquisa na internet', description='\n        Search the web using TavilyAPI.\n        Recommended for more structured and recent information.\n        ', env_vars=[], args_schema=<class 'crewai.tools.base_tool.Pesquisanainternet'>, result_schema=None, description_updated=False, cache_function=<function _default_cache_function at 0x7fc093436fc0>, result_as_answer=False, max_usage_count=None, tool_failure_policy=None, current_usage_count=0, func=<function SearchTools.search_tavily at 0x7fc05a14d580>, tool_type='crewai.tools.base_tool.Tool')

### Outras ferramentas

In [12]:
class CalculatorTools:
    @tool("Faça um cálculo")
    def calculate(operation):
        """
        Useful to perform any mathematical calculations,
        like sum, minus, multiplication, division, etc.
        The input to this tool should be a mathematical expression, a 
        couple examples are `200*7` or `5000/2*10`
        """
        try:
            return eval(operation)
        except SyntaxError:
            return "Erro: Sintaxe inválida"

CalculatorTools.calculate

Tool(name='Faça um cálculo', description='\n        Useful to perform any mathematical calculations,\n        like sum, minus, multiplication, division, etc.\n        The input to this tool should be a mathematical expression, a \n        couple examples are `200*7` or `5000/2*10`\n        ', env_vars=[], args_schema=<class 'crewai.tools.base_tool.Façaumcálculo'>, result_schema=None, description_updated=False, cache_function=<function _default_cache_function at 0x7fc093436fc0>, result_as_answer=False, max_usage_count=None, tool_failure_policy=None, current_usage_count=0, func=<function CalculatorTools.calculate at 0x7fc06a7000e0>, tool_type='crewai.tools.base_tool.Tool')

## Criação de Agentes

In [13]:
from textwrap import dedent

texto = """
    Este é um texto
    com identação errada.
"""
print(texto)


    Este é um texto
    com identação errada.



### Especialista em informações da cidade

In [15]:
from crewai import Agent

city_info_agent = Agent(
    role = "Especialista em informações da cidade",
    goal = dedent(
        """
            Reunir informações úteis sobre o destino escolhido, com base nos interesses e datas de viagem, ajudando viajantes a entender o contexto geral do local.
        """
    ),
    backstory = dedent(
        """
            Sou um especialista com amplo conhecimento sobre cidades ao redor do mundo,
            capaz de fornecer detalhes atualizados sobre clima, eventos culturais, segurança,
            costumes locais e outros aspectos práticos para quem pretende visitar o local. 
            Sou apaixonado por compartilhar as melhores experiências e 'joias escondidas' do local.
        """
    ),
    llm = gemini,
    tools = [SearchTools.search_tavily],
    verbose = True,
    max_iter = 10,
    allow_delegation = False
)

### Logística e informações da cidade

In [16]:
logistics_expert_agent = Agent(
    role = "Especialista em logística de viagem",
    goal = dedent(
        """
            Identificar as melhores opções logísticas para a viagem, com foco em praticidade, custo-benefício e conforto.
        """
    ),
    backstory = dedent(
        """
            Sou um profissional focado em planejar a logística da viagem: transporte, hospedagem e deslocamento local.
            Tenho conhecimento sobre companhias aéreas, apps de mobilidade, regiões seguras para se hospedar e otimização de trajetos.
        """
    ),
    llm=gemini,
    tools=[
        SearchTools.search_tavily,
        CalculatorTools.calculate
    ],
    verbose=True,
    max_iter=5,
    allow_delegation=False
)

### Especialista em criação de itinerário

In [17]:
itinerary_planner_agent = Agent(
    role="Planejador de itinerário personalizado",
    goal="Criar um roteiro completo com base nas preferências do usuário.",
    backstory=dedent(
        """
        Sou um guia profissional apaixonado por viagens, especialista em organização de itinerários personalizados.
        Minha missão é integrar dados sobre clima, atrações, eventos e logística em uma experiência otimizada para o turista.
        """
    ),
    llm=gemini,
    tools=[SearchTools.search_tavily],
    verbose=True,
    max_iter=5,
    allow_delegation=False
)

## Criação das tarefas (Tasks)

### Task: Coleta de informações da cidade

### Task: Planejamento de logística

### Task: Construção de itinerário final